In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm

from optbinning import OptimalBinning
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
FILE_PATH        = r"D:\KL-lieuvth2\kaggle\input\home-credit-default-risk\master_data.csv"
ID_COL           = "SK_ID_CURR"
TARGET_COL       = "TARGET"

MISSING_THR      = 0.20
INF_THR          = 0.20
IV_THR           = 0.01
IV_MAX           = 10
VIF_THR          = 5          # VIF threshold (replaces GVIF_THR)
CORR_THR         = 0.80
GINI_IMPROVE_THR = 0.005
TEST_SIZE        = 0.30
RANDOM_STATE     = 42
IV_BINS          = 10
EPS              = 1e-6

OUTPUT_CSV          = "master_data_train_test_split.csv"
FEATURE_SHORTLIST   = "feature_shortlist.csv"

## 1. Load data — Train/Test split & save CSV with flag_train_test

In [3]:
# df = pd.read_csv(FILE_PATH)
# df = df.drop(columns=[c for c in ["flag_train_val"] if c in df.columns])
# print("Shape after load:", df.shape)

In [4]:
# # ── Train / Test split right at Step 1 ────────────────────────────────────────
# train_idx, test_idx = train_test_split(
#     df.index,
#     test_size=TEST_SIZE,
#     random_state=RANDOM_STATE,
#     stratify=df[TARGET_COL],
# )

# df["flag_train_test"] = "test"
# df.loc[train_idx, "flag_train_test"] = "train"

# # Save full data with flag to CSV
# df.to_csv(OUTPUT_CSV, index=False)
# print(f"Saved: {OUTPUT_CSV}")
# print(f"Train: {(df['flag_train_test']=='train').sum():,}  |  Test: {(df['flag_train_test']=='test').sum():,}")
# print("Shape:", df.shape)

In [5]:
df = pd.read_csv(OUTPUT_CSV).iloc[:, 1:]

In [6]:
df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,CC_BALANCE_DIFF1_MEAN,CC_BALANCE_DIFF2_MEAN,CC_BALANCE_DIFF3_MEAN,CC_BALANCE_DIFF6_MEAN,HAS_BUREAU_HISTORY,HAS_PREV_APPLICATION,HAS_POS_HISTORY,HAS_INSTALLMENT_HISTORY,HAS_CREDIT_CARD,flag_train_test
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.0,NaN,0,1,1,1,1,test
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,NaN,NaN,NaN,NaN,0,1,1,1,0,test
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,NaN,NaN,NaN,NaN,0,1,1,1,0,train
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,NaN,NaN,NaN,NaN,1,1,1,1,0,train


In [7]:
# separate features, target
list_features = [col for col in df.columns if col not in ['TARGET','SK_ID_CURR']]
# features that starts with EXT_
# list_ext_features = [col for col in df.columns if col.startswith('EXT_')]
# list ethical features
list_ethical_features = [col for col in df.columns if 'GENDER' in col]
# list flag document features
list_document_features = [col for col in df.columns if 'FLAG_DOCUMENT' in col]

In [8]:
cols_to_drop = list(set(list_ethical_features + list_document_features))
df = df.drop(columns=cols_to_drop, errors='ignore')

In [9]:
print("Shape after dropping columns:", df.shape)

Shape after dropping columns: (307511, 238)


## 2. Drop high-missing & high-infinity features

In [10]:
# Work on df without the flag column
flag_col = df["flag_train_test"].copy()
df = df.drop(columns=["flag_train_test"])

# ============================================================
# Missing statistics
# ============================================================
miss_count = df.isna().sum()
miss_rate  = df.isna().mean()

missing_stats = pd.DataFrame({
    "missing_count": miss_count,
    "missing_rate": miss_rate
}).sort_values("missing_rate", ascending=False)

drop_missing = missing_stats.loc[
    missing_stats["missing_rate"] > MISSING_THR
].index.tolist()

df = df.drop(columns=drop_missing)


# ============================================================
# Infinity statistics
# ============================================================
num_cols = df.select_dtypes(include=np.number).columns

inf_count = np.isinf(df[num_cols]).sum()
inf_rate  = np.isinf(df[num_cols]).mean()

inf_stats = pd.DataFrame({
    "inf_count": inf_count,
    "inf_rate": inf_rate
}).sort_values("inf_rate", ascending=False)

drop_inf = inf_stats.loc[
    inf_stats["inf_rate"] > INF_THR
].index.tolist()

df = df.drop(columns=drop_inf)


# ============================================================
# Re-attach flag
# ============================================================
df["flag_train_test"] = flag_col


# ============================================================
# Summary
# ============================================================
print(f"Dropped missing : {len(drop_missing)}")
print(f"Dropped infinity: {len(drop_inf)}")
print(f"Remaining       : {df.shape[1] - 1}")
print("Shape:", df.shape)

print("\nTop missing features:")
print(missing_stats.head(10))

print("\nTop infinity features:")
print(inf_stats.head(10))

Dropped missing : 59
Dropped infinity: 0
Remaining       : 178
Shape: (307511, 179)

Top missing features:
                                    missing_count  missing_rate
CC_DRAWINGS_PAYMENT_DPD_CORR_MEAN          298365      0.970258
CC_UTIL_DPD_CORR_MEAN                      289701      0.942083
POS_PAID_DPD_CORR_MEAN                     270904      0.880957
CC_PAYMENT_TOTAL_TO_MIN_RATIO_MEAN         248232      0.807230
CC_DRAWINGS_PAYMENT_RATIO_MEAN             246824      0.802651
CC_UTIL_DIFF6_MEAN                         232661      0.756594
CC_BALANCE_DIFF6_MEAN                      230317      0.748972
CC_UTIL_DIFF3_MEAN                         225020      0.731746
CC_UTIL_DIFF2_MEAN                         223367      0.726371
CC_BALANCE_DIFF3_MEAN                      223307      0.726176

Top infinity features:
                            inf_count  inf_rate
SK_ID_CURR                          0       0.0
TARGET                              0       0.0
CNT_CHILDREN         

In [11]:
inf_stats.to_csv("inf_stats.csv")
missing_stats.to_csv("missing_stats.csv")

## 3. Information Value (IV) via AutoBinning on full data — then compute df_woe

In [12]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]
y_all    = df[TARGET_COL].values

# ── Fit OptimalBinning on full data, extract IV and WoE in one pass ────────────
binners   = {}   # fitted OptimalBinning objects
woe_cols  = {}   # WoE-transformed arrays
iv_rows   = []   # IV per feature

binning_tables = {}

for col in features:
    dtype = "numerical" if pd.api.types.is_numeric_dtype(df[col]) else "categorical"
    ob = OptimalBinning(
        name=col,
        dtype=dtype,
        solver="cp",
        max_n_bins=IV_BINS
    )

    ob.fit(df[col].values, y_all)

    binners[col] = ob
    woe_cols[col] = ob.transform(df[col].values, metric="woe")

    # Lưu binning table
    bt = ob.binning_table.build()
    binning_tables[col] = bt.copy()

    try:
        iv = bt.loc[bt.index != "Totals", "IV"].sum()
    except Exception:
        iv = np.nan

    iv_rows.append({"feature": col, "iv": iv})

# ── IV table ───────────────────────────────────────────────────────────────────
iv_table = (
    pd.DataFrame(iv_rows)
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

bins_iv = [-np.inf, 0.01, 0.02, 0.10, 0.30, 0.50, np.inf]
labels  = ["<0.01", "0.01-0.02", "0.02-0.10", "0.10-0.30", "0.30-0.50", ">0.50"]
iv_table["group"] = pd.cut(iv_table["iv"], bins=bins_iv, labels=labels)
print(iv_table["group"].value_counts().reindex(labels).fillna(0).astype(int))

# ── Filter by IV ───────────────────────────────────────────────────────────────
keep_iv = iv_table.loc[
    iv_table["iv"].between(IV_THR, IV_MAX),
    "feature"
].tolist()
print(f"\nFeatures after IV filter: {len(keep_iv)}")

# ── Build df_woe on full data (IV-passing features only) ──────────────────────
df_woe = pd.DataFrame({col: woe_cols[col] for col in keep_iv}, index=df.index)
df_woe.insert(0, ID_COL,          df[ID_COL].values)
df_woe.insert(1, TARGET_COL,      df[TARGET_COL].values)
df_woe.insert(2, "flag_train_test", df["flag_train_test"].values)

print("df_woe shape:", df_woe.shape)
df_woe.head(3)

group
<0.01        38
0.01-0.02    49
0.02-0.10    81
0.10-0.30     3
0.30-0.50     4
>0.50         1
Name: count, dtype: int64

Features after IV filter: 138
df_woe shape: (307511, 141)


,SK_ID_CURR,TARGET,flag_train_test,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_MAX,EXT_SOURCE_3,EXT_SOURCE_2,EMPLOY_YEARS,DAYS_EMPLOYED,...,HOUSING_LIVINGAPARTMENTS_WAS_MISSING,AMT_INCOME_TOTAL,HAS_BUREAU_HISTORY,BUREAU_HAS_BAD_DEBT,HOUSING_NONLIVINGAPARTMENTS_WAS_MISSING,HOUSING_COMMONAREA_WAS_MISSING,EXT_SOURCE_NAN_COUNT,BUREAU_DEBT_SUM,PREV_APP_COUNT,FLAG_WORK_PHONE
0,100002,1,train,-1.372552,-1.184253,-1.229897,-1.176614,-0.331357,-0.343809,-0.343809,...,0.171513,0.008517,0.047117,0.047117,0.167647,0.168085,0.109278,-0.041173,0.018093,0.053434
1,100003,0,train,-0.061017,-0.112749,0.162243,0.000000,0.396376,-0.252375,-0.252375,...,0.171513,0.146626,0.047117,0.047117,0.167647,0.168085,-0.014111,0.113854,0.018093,0.053434
2,100004,0,train,1.022666,0.786998,0.696196,0.913839,0.206619,-0.371972,-0.371972,...,-0.071716,0.036022,0.047117,0.047117,-0.066903,-0.065718,-0.014111,0.113854,0.018093,-0.193464


In [13]:
iv_table

,feature,iv,group
0,EXT_SOURCE_MEAN,0.623410,>0.50
1,EXT_SOURCE_MIN,0.474334,0.30-0.50
2,EXT_SOURCE_MAX,0.451813,0.30-0.50
3,EXT_SOURCE_3,0.335228,0.30-0.50
4,EXT_SOURCE_2,0.320699,0.30-0.50
...,...,...,...
171,LIVE_REGION_NOT_WORK_REGION,0.000000,<0.01
172,FLAG_CONT_MOBILE,0.000000,<0.01
173,BUREAU_N_BAD_DEBT,0.000000,<0.01
174,BUREAU_N_SOLD,0.000000,<0.01


In [14]:
iv_table.to_csv("iv_table.csv")

In [15]:
df_woe.to_csv("df_woe_long_list.csv")

## 4. Correlation on df_woe — keep higher-IV feature in each correlated pair

In [16]:
# WoE features
woe_features = [
    c for c in df_woe.columns
    if c not in [ID_COL, TARGET_COL, "flag_train_test"]
]

# Correlation matrix
corr = df_woe[woe_features].corr().abs()
corr.to_csv("correlation_matrix.csv")

# High-correlation pairs
pairs = [
    {
        "feature_1": corr.columns[i],
        "feature_2": corr.columns[j],
        "corr": corr.iloc[i, j]
    }
    for i in range(len(corr.columns))
    for j in range(i + 1, len(corr.columns))
    if corr.iloc[i, j] > CORR_THR
]

pairs = pd.DataFrame(pairs).sort_values(
    "corr", ascending=False
).reset_index(drop=True)

pairs.to_csv("high_corr_pairs.csv", index=False)

# Drop lower-IV feature
iv_dict = iv_table.set_index("feature")["iv"].to_dict()

drop_set = set()
drop_log = []

for _, row in pairs.iterrows():
    f1, f2 = row["feature_1"], row["feature_2"]

    if f1 in drop_set or f2 in drop_set:
        continue

    iv1 = iv_dict.get(f1, 0)
    iv2 = iv_dict.get(f2, 0)

    dropped = f2 if iv1 >= iv2 else f1
    kept = f1 if dropped == f2 else f2

    drop_set.add(dropped)

    drop_log.append({
        "feature_1": f1,
        "feature_2": f2,
        "corr": row["corr"],
        "iv_1": iv1,
        "iv_2": iv2,
        "kept": kept,
        "dropped": dropped
    })

# Save drop log
pd.DataFrame(drop_log).to_csv(
    "correlation_drop_log.csv", index=False
)

# Drop features
df_woe = df_woe.drop(
    columns=[c for c in drop_set if c in df_woe.columns]
)

# Refresh IV table
surviving = [
    c for c in df_woe.columns
    if c not in [ID_COL, TARGET_COL, "flag_train_test"]
]

iv_table = (
    iv_table[iv_table["feature"].isin(surviving)]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

print(f"High-corr pairs: {len(pairs)}")
print(f"Dropped by correlation: {len(drop_set)}")
print(f"Remaining features: {len(surviving)}")

High-corr pairs: 139
Dropped by correlation: 54
Remaining features: 84


## 5. VIF on df_woe — drop multicollinear features

In [17]:
import time

woe_features = [
    c for c in df_woe.columns
    if c not in [ID_COL, TARGET_COL, "flag_train_test"]
]

# df_woe columns are already numeric WoE values
X_vif = df_woe[woe_features].copy()

# Handle inf / NaN
X_vif = (
    X_vif
    .replace([np.inf, -np.inf], np.nan)
    .fillna(X_vif.median())
)

print(f"Calculating VIF for {X_vif.shape[1]} WoE features...")
t0 = time.time()

# ============================================================
# 1. Calculate VIF ONCE for all features
# ============================================================
X_tmp = sm.add_constant(X_vif, has_constant="add")

vif_vals = [
    variance_inflation_factor(X_tmp.values, i + 1)
    for i in range(len(woe_features))
]

vif_table = pd.DataFrame({
    "feature": woe_features,
    "vif": vif_vals
}).sort_values(
    "vif",
    ascending=False
).reset_index(drop=True)

print(f"Finished VIF in {time.time() - t0:.1f}s")

# ============================================================
# 2. Drop ALL features exceeding threshold — only once
# ============================================================
drop_vif = vif_table.loc[
    vif_table["vif"] > VIF_THR,
    "feature"
].tolist()

remaining = [
    c for c in woe_features
    if c not in drop_vif
]

print(f"\nVIF threshold: {VIF_THR}")
print(f"Features before VIF filter: {len(woe_features)}")
print(f"Dropped by VIF: {len(drop_vif)}")
print(f"Remaining features: {len(remaining)}")

if drop_vif:
    print("\nDropped features:")
    print(vif_table[vif_table.feature.isin(drop_vif)].to_string(index=False))

# ============================================================
# 3. Remove features from df_woe
# ============================================================
df_woe = df_woe.drop(
    columns=[c for c in drop_vif if c in df_woe.columns]
)

print("\ndf_woe shape after VIF filter:", df_woe.shape)

# ============================================================
# 4. Refresh IV table
# ============================================================
surviving = [
    c for c in df_woe.columns
    if c not in [ID_COL, TARGET_COL, "flag_train_test"]
]

iv_table = (
    iv_table[
        iv_table["feature"].isin(surviving)
    ]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)

print("\nIV table after VIF filter:")
print(iv_table.to_string(index=False))

Calculating VIF for 84 WoE features...
Finished VIF in 411.3s

VIF threshold: 5
Features before VIF filter: 84
Dropped by VIF: 7
Remaining features: 77

Dropped features:
                        feature       vif
                BUREAU_N_CLOSED 10.619435
      HOUSING_INFO_COMPLETENESS  5.839367
          BUREAU_CREDIT_SUM_SUM  5.792676
                 BUREAU_N_LOANS  5.741245
HOUSING_YEARS_BUILD_WAS_MISSING  5.661067
            BUREAU_ACTIVE_RATIO  5.087099
                EXT_SOURCE_MEAN  5.003347

df_woe shape after VIF filter: (307511, 80)

IV table after VIF filter:
                           feature       iv     group
                      EXT_SOURCE_3 0.335228 0.30-0.50
                      EXT_SOURCE_2 0.320699 0.30-0.50
                      EMPLOY_YEARS 0.114054 0.10-0.30
          BUREAU_DEBT_CREDIT_RATIO 0.103937 0.10-0.30
                   AMT_GOODS_PRICE 0.092037 0.02-0.10
                               AGE 0.086928 0.02-0.10
            BUREAU_DAYS_CREDIT_MAX 0.08298

In [18]:
vif_table.to_csv("vif_table.csv")

## 6. Save feature shortlist before stepwise

In [19]:
shortlist_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

shortlist_df = (
    iv_table[iv_table.feature.isin(shortlist_features)]
    [["feature", "iv", "group"]]
    .sort_values("iv", ascending=False)
    .reset_index(drop=True)
)
shortlist_df.to_csv(FEATURE_SHORTLIST, index=False)

print(f"Feature shortlist saved to: {FEATURE_SHORTLIST}")
print(f"Total features in shortlist: {len(shortlist_features)}")
print(shortlist_df.to_string(index=False))

Feature shortlist saved to: feature_shortlist.csv
Total features in shortlist: 77
                           feature       iv     group
                      EXT_SOURCE_3 0.335228 0.30-0.50
                      EXT_SOURCE_2 0.320699 0.30-0.50
                      EMPLOY_YEARS 0.114054 0.10-0.30
          BUREAU_DEBT_CREDIT_RATIO 0.103937 0.10-0.30
                   AMT_GOODS_PRICE 0.092037 0.02-0.10
                               AGE 0.086928 0.02-0.10
            BUREAU_DAYS_CREDIT_MAX 0.082983 0.02-0.10
            BUREAU_DAYS_CREDIT_MIN 0.079678 0.02-0.10
                GOODS_CREDIT_RATIO 0.076063 0.02-0.10
PREV_CREDIT_APPLICATION_RATIO_MEAN 0.072437 0.02-0.10
                 PREV_REFUSAL_RATE 0.069869 0.02-0.10
                PREV_APPROVAL_RATE 0.064407 0.02-0.10
                     INST_PCT_LATE 0.063627 0.02-0.10
                        AMT_CREDIT 0.059368 0.02-0.10
         ORGANIZATION_TYPE_GROUPED 0.058196 0.02-0.10
                  NAME_INCOME_TYPE 0.057869 0.02-0.10


## 7. Train / Test split from df_woe

In [20]:
train_woe = df_woe[df_woe["flag_train_test"] == "train"].reset_index(drop=True)
test_woe  = df_woe[df_woe["flag_train_test"] == "test"].reset_index(drop=True)

model_features = [c for c in df_woe.columns if c not in [ID_COL, TARGET_COL, "flag_train_test"]]

X_train = train_woe[model_features]
y_train = train_woe[TARGET_COL]
X_test  = test_woe[model_features]
y_test  = test_woe[TARGET_COL]

print(f"Train: {train_woe.shape}  |  Test: {test_woe.shape}")

Train: (215257, 80)  |  Test: (92254, 80)


## 8. Stepwise Logistic Regression (forward, Gini-based)

In [21]:
from numpy.linalg import matrix_rank

def fit_logit(X, y):
    X = X.replace([np.inf, -np.inf], np.nan)
    if X.isna().any().any():
        raise ValueError("X contains NaN.")
    X_c = sm.add_constant(X, has_constant="add")
    if matrix_rank(X_c) < X_c.shape[1]:
        return None
    return sm.Logit(y, X_c).fit(disp=False, method="newton")


def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1


# Use shortlist order (sorted by IV) as candidates
candidates   = [f for f in shortlist_df["feature"].tolist() if f in X_train.columns]
selected     = []
current_gini = 0.0
log_rows     = []

for i, feat in enumerate(candidates, 1):
    print(f"[{i}/{len(candidates)}] Testing: {feat}", flush=True)

    trial_feats = selected + [feat]
    model_try   = fit_logit(X_train[trial_feats], y_train)

    if model_try is None:
        print("    -> skipped (singular matrix)", flush=True)
        log_rows.append({"feature": feat, "gini_train": None,
                         "improvement": None, "accepted": False, "note": "singular"})
        continue

    prob_train  = model_try.predict(sm.add_constant(X_train[trial_feats], has_constant="add"))
    new_gini    = gini(y_train, prob_train)
    improvement = new_gini - current_gini
    accept      = improvement >= GINI_IMPROVE_THR

    print(f"    Gini={new_gini:.4f}  Improve={improvement:.4f}  {'ACCEPT' if accept else 'REJECT'}",
          flush=True)

    log_rows.append({"feature": feat, "gini_train": round(new_gini, 4),
                     "improvement": round(improvement, 4), "accepted": accept, "note": ""})

    if accept:
        selected     = trial_feats
        current_gini = new_gini

step_log = pd.DataFrame(log_rows)
print(step_log.to_string(index=False))
print(f"\nFinal model features ({len(selected)}): {selected}")

[1/77] Testing: EXT_SOURCE_3
    Gini=0.3154  Improve=0.3154  ACCEPT
[2/77] Testing: EXT_SOURCE_2
    Gini=0.4221  Improve=0.1067  ACCEPT
[3/77] Testing: EMPLOY_YEARS
    Gini=0.4351  Improve=0.0131  ACCEPT
[4/77] Testing: BUREAU_DEBT_CREDIT_RATIO
    Gini=0.4403  Improve=0.0052  ACCEPT
[5/77] Testing: AMT_GOODS_PRICE
    Gini=0.4517  Improve=0.0114  ACCEPT
[6/77] Testing: AGE
    Gini=0.4543  Improve=0.0026  REJECT
[7/77] Testing: BUREAU_DAYS_CREDIT_MAX
    Gini=0.4518  Improve=0.0001  REJECT
[8/77] Testing: BUREAU_DAYS_CREDIT_MIN
    Gini=0.4517  Improve=0.0001  REJECT
[9/77] Testing: GOODS_CREDIT_RATIO
    Gini=0.4606  Improve=0.0089  ACCEPT
[10/77] Testing: PREV_CREDIT_APPLICATION_RATIO_MEAN
    Gini=0.4692  Improve=0.0086  ACCEPT
[11/77] Testing: PREV_REFUSAL_RATE
    Gini=0.4732  Improve=0.0040  REJECT
[12/77] Testing: PREV_APPROVAL_RATE
    Gini=0.4716  Improve=0.0025  REJECT
[13/77] Testing: INST_PCT_LATE
    Gini=0.4785  Improve=0.0094  ACCEPT
[14/77] Testing: AMT_CREDIT
    G

## 9. Final model & Test Gini

In [22]:
final_model = fit_logit(X_train[selected], y_train)
print(final_model.summary())

prob_train_final = final_model.predict(sm.add_constant(X_train[selected]))
prob_test_final  = final_model.predict(sm.add_constant(X_test[selected]))

gini_train = gini(y_train, prob_train_final)
gini_test  = gini(y_test,  prob_test_final)

print(f"\nGini Train : {gini_train:.4f}")
print(f"Gini Test  : {gini_test:.4f}")

                           Logit Regression Results                           
Dep. Variable:                 TARGET   No. Observations:               215257
Model:                          Logit   Df Residuals:                   215246
Method:                           MLE   Df Model:                           10
Date:                Sun, 16 Aug 2026   Pseudo R-squ.:                  0.1081
Time:                        23:06:15   Log-Likelihood:                -53861.
converged:                       True   LL-Null:                       -60388.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                 -2.4487      0.009   -268.086      0.000      -2.467      -2.431
EXT_SOURCE_3                          -0.7352      0.015   

In [23]:
import pickle

with open("binning_tables.pkl", "wb") as f:
    pickle.dump(binning_tables, f)
with open("binners.pkl", "wb") as f:
    pickle.dump(binners, f)
with open("model.pkl", "wb") as f:
    pickle.dump(final_model, f)